# Chapter 10 Companion Notebook: Tree-Based Models: Housing Price Random Forest

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch10_Housing_Price_Random_Forest_Regression.ipynb)

This notebook accompanies Chapter 10 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
- Click "Upload" and select this file and the data file.

# Housing price: Bagging, Random forests, Boosting

### Use "Housing price.csv".
- id: Unique identifier for each property  
- list: Date of property listing (YYYYMMDD)
- yr_built: The year the property was originally built.  
- yr_renovated: The year the property was last renovated (0 indicates no renovation).  
- price: Property price
- bedrooms: Number of bedrooms  
- bathrooms: Number of bathrooms  
- sqft_living: Living area size in square feet  
- sqft_lot: Lot size in square feet  
- floors: Number of floors  
- waterfront: 1 if property has waterfront view
- view: Quality level of property view (0 to 4)  
- condition: the overall condition of the property (1 to 5).  
- grade: the construction and design quality of the property (1-13).  

In [ ]:
# pip install xgboost

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, GridSearchCV
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from matplotlib.pyplot import subplots
import xgboost
import matplotlib.pyplot as plt

### Prepare data

In [ ]:
df = pd.read_csv('Housing price.csv')
df.head()

### 1. Create the following variables: 'years since built' and 'years since renovated'.

In [ ]:
from datetime import datetime

# Convert the "date" column to a datetime format
df['list'] = pd.to_datetime(df['list'], format='%Y%m%d')

# Extract the "list year" from the date column
df['list_year'] = df['list'].dt.year

# Calculate "years since renovated" and "years since built"
df['years_since_renovated'] = df.apply(lambda row: row['list_year'] - row['yr_renovated']
            if row['yr_renovated'] > 0 else row['list_year'] - row['yr_built'], axis=1)
df['years_since_built'] = df['list_year'] - df['yr_built']
df.head()

### 2. Define dependent and independent variables. Divide the data into 75% training and 25% test set (use random_state=15).
- Dependent variable: price
- Independent variables: 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade', 'years_since_renovated', 'years_since_built'

In [ ]:
# Split the data into features and target variable
y = df.price
x = df[['bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'waterfront', 
        'view', 'condition', 'grade', 'years_since_renovated', 'years_since_built']]

# Splitting the dataset into the Training set and Test set
xtrain, xtest, ytrain, ytest = train_test_split(x, y, random_state=15)

- `x = df.drop('value', axis=1)` creates a new DataFrame `x` by dropping the column 'value'.
    - `axis=1` indicates that the operation should be performed along columns (i.e., dropping a column).
- `y = df.value` assigns the values from the 'value' column to a variable `y`.
- `xtrain, xtest, ytrain, ytest = train_test_split()` splits the data into training and testing sets.
- `xtrain = StandardScaler().fit_transform(xtrain)` standardizes the training features (`xtrain`). It involves scaling the features to have a mean of 0 and a standard deviation of 1.
    - `fit_transform` fitS the scaler to the training data (`xtrain`) and then transform it.

### Decision trees

In [ ]:
m1 = DecisionTreeRegressor(max_depth=3).fit(xtrain, ytrain)
pred1 = m1.predict(xtest)
mse1 = mean_squared_error(ytest, pred1)  # MSE=np.mean((ytest - pred1)**2)
mse1

- `max_depth`: The maximum depth of each individual decision tree, which controls the complexity of the model.
- `fit`: trains the model using DecisionTreeRegressor on xtrain and ytrain sets.
    - The prediction for a given data point is determined by the leaf node it ends up in. Each data point in the test data traverses the decision tree based on the conditions in each node and lands in a specific leaf node. The value associated with that leaf node is the prediction for that data point.
- `pred1 = m1.predict(xtest)`: uses the trained model `m1` to make predictions on the test data (`xtest`) and stores the predicted values in the variable `pred1`.
- `mse1 = mean_squared_error(ytest, pred1)`: calculates the Mean Squared Error (MSE) between the true target values (`ytest`) and the predicted values (`pred1`) from the model.

### Bagging
- Bagging is a special case of a random forest with $m=p$. Therefore, `RandomForestRegressor()` can be used to perform both bagging and random forests.

In [ ]:
m2 = RandomForestRegressor(max_features=xtrain.shape[1], random_state=1).fit(xtrain, ytrain)
pred2 = m2.predict(xtest)
mse2 = mean_squared_error(ytest, pred2)
mse2

- `'max_features=xtrain.shape[1]'` sets the maximum number of features that the random forest will consider when making splits during tree construction.
- `'xtrain.shape[1]'` represents to the number of features in the xtrain dataset.
    - `shape[0]` corresponds to the number of rows or the size of the first dimension.
    - `shape[1]` corresponds to the number of columns or the size of the second dimension.
- `fit(xtrain, ytrain)` trains the random forest regression model using the training data.

In [ ]:
# % change in MSE

(mse2-mse1)/mse1

- The test MSE for the bagged regression tree is about 30% lower than that of a single tree

In [ ]:
# Increase the number of trees grown

m3 = RandomForestRegressor(n_estimators=400, random_state=1).fit(xtrain, ytrain)
pred3 = m3.predict(xtest)
mse3 = mean_squared_error(ytest, pred3)
mse3

- `'n_estimators'`: The number of trees in the forest (default=100)
- The increased number of trees did not improve predictive performance in this case.

### Random forests

In [ ]:
m4 = RandomForestRegressor(max_features=5, random_state=1).fit(xtrain, ytrain)
pred4 = m4.predict(xtest)
mean_squared_error(ytest, pred4)

- The test MSE for the random forest regression tree is slightly lower than that of bagging.

In [ ]:
# Variable importance

m4.feature_importances_

- `'m4.feature_importances_'` retrieves the feature importances from a trained Random Forest Regressor model `m4`, providing the significance of each feature in making predictions.
- Variable importance is a measure of the total decrease in node impurity that results from splits over each variable, averaged over all trees.

In [ ]:
# Combine with feature names

imp = pd.DataFrame({'importance': m4.feature_importances_}, index=x.columns)
imp.sort_values(by='importance', ascending=False)

- Across all of the trees considered in the random forest, the wealth level of the community (`low`: percentage of the lower status of the populationlstat) and the ouse size (`Room`: average number of rooms per dwelling) are by far the two most important variables.

- `imp = pd.DataFrame({'importance': m4.feature_importances_}, index=x.columns)`
    - `pd.DataFrame()` constructs a new DataFrame `imp` with two columns: `importance` and `feature`.
    - `{'importance': m4.feature_importances_}` assigns the feature importances to the 'importance' column.
    - `index = x.columns` retrieves the column names (feature names) of the DataFrame `x` and use them as the index (row labels) of the dataframe `imp` to create a mapping between feature names and their importances.
- `'imp.sort_values(by='importance', ascending=False)'`
   - This line sorts the `imp` DataFrame by the `importance` column in descending order. As a result, the features with the highest importances will appear at the top.

In [ ]:
# Variable importance graph

pd.Series(m4.feature_importances_, index=x.columns).sort_values().plot.barh(figsize=(5,4))
plt.xlabel('Importance')
plt.title('Feature importances')

- `'pd.Series(m4.feature_importances_, index=x.columns)'` creates a pandas Series object. It's a one-dimensional labeled array that can hold data of various types.
  - `'.sort_values()'`: Sorts the Series in ascending order based on the importance scores.
- `'.plot.barh(figsize=())'`: creates a horizontal bar plot of the sorted Series.

### Boosting

In [ ]:
m5 = GradientBoostingRegressor(n_estimators=4000, learning_rate=0.01, max_depth=3, random_state=1).fit(xtrain, ytrain)
pred5 = m5.predict(xtest)
mean_squared_error(ytest, pred5)

- `GradientBoostingRegressor()`: Gradient Boosting is an ensemble learning method that builds a predictive model in the form of an ensemble of weak learners, typically decision trees. It does this in a sequential manner, with each new tree attempting to correct the errors made by the previous ones.
    - It uses gradient descent optimization to iteratively minimize a loss function, such as mean squared error (MSE), with respect to the predictions of the ensemble.
    - `'learning_rate'` controls the step size during gradient descent. It shrinks the contribution of each tree by `learning_rate`(default=0.1).    

In [ ]:
# Try different learning rates

m6 = GradientBoostingRegressor(n_estimators=4000, learning_rate=0.2, max_depth=3, random_state=1).fit(xtrain, ytrain)
pred6 = m5.predict(xtest)
mean_squared_error(ytest, pred6)

`learning_rate`: float, default=0.1
    Learning rate shrinks the contribution of each tree by `learning_rate`.
    There is a trade-off between learning_rate and n_estimators.
    Values must be in the range `[0.0, inf)`

### XGBoost

In [ ]:
m7 = xgboost.XGBRegressor(n_estimators=4000, max_depth=3, learning_rate=0.01).fit(xtrain, ytrain)
pred7 = m7.predict(xtest)
mean_squared_error(ytest, pred7)

- XGBoost further reduces the test MSE.

### AdaBoost

In [ ]:
m8 = AdaBoostRegressor(n_estimators=4000, learning_rate=0.01).fit(xtrain, ytrain)
pred8 = m8.predict(xtest)
mean_squared_error(ytest, pred8)

- XGBoost performs bettern than AdaBoost.

### Visualize the test error for each boosting iteration (tree)

In [ ]:
# Visualize the test error as the number of boosting iterations (trees) increases

testerror = np.zeros_like(m5.train_score_)
for idx, y_ in enumerate(m5.staged_predict(xtest)):
    testerror[idx] = np.mean((ytest - y_)**2)

plot_idx = range(m5.train_score_.shape[0])
ax = subplots(figsize=(5,5))[1]
ax.plot(plot_idx, m5.train_score_, label='Training')
ax.plot(plot_idx, testerror, label='Test')
ax.set_xlabel('Number of trees')
ax.set_ylabel('Test MSE')
ax.legend()

- `for idx, y_ in enumerate(m5.staged_predict(xtest)):`  
    - `enumerate()` iterates over an iterable (such as a list, tuple, or string) and keeps track of the index (position) of the current item in the iterable.
    - `staged_predict()` represents the model's predictions at each boosting stage (i.e., each decision tree).
    - `y_` is the predicted values at the current boosting stage. It's the output of `m5.staged_predict(xtest)` for the specific iteration represented by `idx`.
- `testerror[idx] = np.mean((ytest - y_)**2)` calculates the MSE for the test data at each boosting stage and stores it in the `testerror` array at the corresponding index `idx`.     
- `plot_idx = np.arange(m5.train_score_.shape[0])`: Creates an array `plot_idx` that represents the range of boosting stages (from 0 to the total number of stages).
- `ax = subplots(figsize=(8,8))[1]`
    - `subplots()` creates one or more subplots within a figure.
    - `[1]` is used to access the second element of the tuple returned by `subplots()`.
        - The first element of the tuple is the figure object, and the second element is an array of axes objects.
        - By selecting `[1]`, you are assigning the second element (an axes object) to the variable `ax`.
    - `figsize=()` is an optional argument that sets the size of the figure.
- `ax.plot(plot_idx, m5.train_score_, label='Training')`: Plots the training MSE (m5.train_score_) over the boosting stages (plot_idx) and labels it as 'Training'.
- `ax.legend()`: Adds a legend to the plot to distinguish between the training and test error curves.

In [ ]:
print(m5.train_score_)
m5.train_score_.shape

- `'m5.train_score_ contains training data MSE values for each stage (tree) of the gradient boosting process.
- `'m5.train_score_.shape'`: Retrieves the shape (i.e., dimensions) of the train_score_ attribute.

In [ ]:
print(m5.train_score_.shape[0])
range(m5.train_score_.shape[0])

- `m5.train_score_.shape[0]` returns the number of elements in the first dimension(`[0]`) of `train_score_`.
- `range(start, end, step)` generates an array of evenly spaced values within a specified range.
    - If start is omitted, it starts from 0 (default).
    - If step is omitted, it uses step size 1 (default).
    - That is, `range(m5.train_score_.shape[0])` is the same as `range(0, m5.train_score_.shape[0], 1)`.
- `range()' accepts only integers.
- `np.arange()` accepts both integer and floating-point values for the start, stop, and step size.

In [ ]:
# np.zeros_like()

print(np.zeros_like(m5.train_score_))
np.zeros_like(m5.train_score_).shape

- `'np.zeros_like(m5.train_score_)'`: creates a NumPy array filled with zeros and has the same shape (dimensions) as `train_score_`.

In [ ]:
# enumerate() example

fruit = ["apple", "banana", "cherry"]
for index, fruit in enumerate(fruit):
    print(f"{index}: {fruit}")

- `'for index, fruit in enumerate(fruit):`': This line starts a for loop that iterates through the elements of the fruit list. 
    - `enumerate()` iterates through the list while simultaneously obtaining both the index and the value of each element.
    - `index` will store the index of the current fruit, and `fruit` will store the value (the name of the fruit).
- `print(f"{index}: {fruit}")`: Inside the loop, it prints a formatted string that includes the index and the corresponding fruit.
    - `{index}` and `{fruit}` are placeholders that will be replaced by the values of the index and fruit variables, respectively.